# व्यय दावे का विश्लेषण

यह नोटबुक दिखाता है कि कैसे एजेंट बनाए जाते हैं जो प्लगइन्स का उपयोग कर स्थानीय रसीद छवियों से यात्रा व्यय को संसाधित करते हैं, एक व्यय दावा ईमेल बनाते हैं, और व्यय डेटा को पाई चार्ट के माध्यम से विज़ुअलाइज़ करते हैं। एजेंट कार्य संदर्भ के अनुसार गतिशील रूप से फ़ंक्शन चुनते हैं।

चरण:
1. OCR एजेंट स्थानीय रसीद छवि को संसाधित करता है और यात्रा व्यय डेटा निकालता है।
2. ईमेल एजेंट एक व्यय दावा ईमेल बनाता है।

### यात्रा व्यय परिदृश्य का उदाहरण:
कल्पना करें कि आप एक कर्मचारी हैं जो व्यवसाय बैठक के लिए दूसरे शहर में यात्रा कर रहे हैं। आपकी कंपनी की नीति है कि सभी उचित यात्रा-संबंधित व्यय की प्रतिपूर्ति की जाए। यहाँ संभावित यात्रा व्ययों का विवरण है:
- परिवहन:
आपके घर शहर से गंतव्य शहर के लिए राउंड ट्रिप के लिए हवाई किराया।
हवाई अड्डे तक और वापस टैक्सी या राइड-हेलिंग सेवाएँ।
गंतव्य शहर में स्थानीय परिवहन (जैसे सार्वजनिक परिवहन, रेंटल कार, या टैक्सी)।

- आवास:
बैठक स्थल के पास एक मध्यम श्रेणी के व्यावसायिक होटल में तीन रातों का प्रवास।

- भोजन:
कंपनी की प्रति दिवस नीति के अनुसार नाश्ता, दोपहर का भोजन और रात के भोजन के लिए दैनिक भोजन भत्ता।

- विविध व्यय:
हवाई अड्डे पर पार्किंग शुल्क।
होटल में इंटरनेट एक्सेस शुल्क।
टिप या छोटे सेवा शुल्क।

- दस्तावेज़ीकरण:
आप सभी रसीदें (उड़ान, टैक्सी, होटल, भोजन, आदि) और प्रतिपूर्ति के लिए एक पूर्ण व्यय रिपोर्ट जमा करते हैं।


## आवश्यक पुस्तकालय आयात करें

नोटबुक के लिए आवश्यक पुस्तकालयों और मॉड्यूल को आयात करें।


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## व्यय मॉडल परिभाषित करें

 व्यक्तिगत व्यय के लिए एक Pydantic मॉडल बनाएं और एक ExpenseFormatter क्लास बनाएँ जो उपयोगकर्ता क्वेरी को संरचित व्यय डेटा में परिवर्तित करे।

 प्रत्येक व्यय इस प्रारूप में प्रदर्शित किया जाएगा:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## टूल्स को परिभाषित करना - ईमेल जनरेट करना

एक टूल फ़ंक्शन बनाएं जो खर्च दावा सबमिट करने के लिए ईमेल जनरेट करे।
- यह टूल Microsoft Agent Framework से `@tool` डेकोरेटर का उपयोग करता है।
- यह खर्चों की कुल राशि की गणना करता है और विवरणों को ईमेल बॉडी में फॉर्मेट करता है।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# रसीद छवियों से यात्रा व्यय निकालने के लिए उपकरण

रसीद छवियों से यात्रा व्यय निकालने के लिए एक उपकरण फ़ंक्शन बनाएं।
- यह उपकरण Microsoft एजेंट फ्रेमवर्क से `@tool` डेकोरेटर का उपयोग करता है।
- यह रसीद छवि को पढ़ता है, इसे बेस64 के रूप में एन्कोड करता है, और एजेंट के विश्लेषण के लिए डेटा URI लौटाता है।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्चों को संसाधित करना

एजेंट्स को परिभाषित करें और उन्हें `WorkflowBuilder` का उपयोग करके एक अनुक्रमिक वर्कफ़्लो में जोड़ें।
- OCR एजेंट रसीद की छवि से संरचित खर्च डेटा निकालता है `load_receipt_image` टूल का उपयोग करके।
- Email एजेंट निकाले गए डेटा को लेता है और एक पेशेवर खर्च दावा ईमेल बनाता है `generate_expense_email` टूल का उपयोग करके।
- `WorkflowBuilder` `add_edge` के साथ एक अनुक्रमिक पाइपलाइन बनाता है: OCR एजेंट → Email एजेंट।


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## मुख्य फ़ंक्शन

अनुक्रमिक कार्यप्रवाह बनाएँ और इसे चलाएँ ताकि रसीद की छवि को संसाधित किया जा सके और व्यय दावा ईमेल उत्पन्न हो सके।


> **नोट:** यह वर्कफ़्लो वर्तमान में रसीद छवि को base64 टेक्स्ट के रूप में पास करता है, जिसे अधिकांश चैट मॉडल (जिसमें gpt-4.1-mini शामिल है) चित्र के रूप में नहीं मानते हैं।
> यह मॉडल संदर्भ विंडो को भी पार कर सकता है। OCR को Azure AI Vision (या किसी अन्य OCR उपकरण) के साथ चलाना वरीयता दें और केवल निकाला गया टेक्स्ट पास करें, या छवि को `image_url` संदेश के रूप में भेजने के लिए पुनः संरचित करें।
> यदि आप केवल संदर्भ त्रुटियों से बचना चाहते हैं, तो एक छोटी रसीद छवि या बड़ी संदर्भ विंडो वाला मॉडल आज़माएं।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
